<a href="https://colab.research.google.com/github/Csahoo009/ML-for-DA/blob/main/T6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files

# Load dataset directly from uploaded CSV
uploaded = files.upload()

import pandas as pd
df = pd.read_csv("indian_house_prices.csv")

# Scale Price to ₹ Lakhs (1 Lakh = 100,000 INR)
df["Price_Lakhs"] = df["Price"] / 100000.0

print(f"Total records: {df.shape[0]}, Total features: {df.shape[1]}")
print(df[["City", "Location", "Area", "No. of Bedrooms", "Resale", "Price_Lakhs"]].head())

Saving indian_house_prices.csv to indian_house_prices.csv
Total records: 32963, Total features: 42
        City                         Location  Area  No. of Bedrooms  Resale  \
0  Bangalore                 JP Nagar Phase 1  3340                4       0   
1  Bangalore       Dasarahalli on Tumkur Road  1045                2       0   
2  Bangalore  Kannur on Thanisandra Main Road  1179                2       0   
3  Bangalore                     Doddanekundi  1675                3       0   
4  Bangalore                          Kengeri  1670                3       0   

   Price_Lakhs  
0       300.00  
1        78.88  
2        48.66  
3        83.58  
4        68.45  


In [ ]:
# ==========================================================
# T6: Logistic Regression Classification Performance Matrix
# ==========================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# 1. Define Feature Sets to Predict 'Resale' (0: New Property, 1: Resale)
feature_subsets = {
    "Core (Area, Price)": ['Area', 'Price_Lakhs'],
    "Extended (+ BHK, Gym)": ['Area', 'Price_Lakhs', 'No. of Bedrooms', 'Gymnasium'],
    "Full (+ Pool, Security, Parking, Lift)": [
        'Area', 'Price_Lakhs', 'No. of Bedrooms', 'Gymnasium',
        'SwimmingPool', '24X7Security', 'CarParking', 'LiftAvailable'
    ]
}

target_col = 'Resale'
y = df[target_col]


In [ ]:
# 2. Evaluation Matrix Pipeline
results = []
trained_models = {}

for name, features in feature_subsets.items():
    X = df[features].fillna(df[features].mean())

    # Stratified Train-Test Split (80/20)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Feature Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train Logistic Regression
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train_scaled, y_train)

    # Predictions & Probabilities
    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]

    # Compute Classification Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)

    # 5-Fold Stratified Cross-Validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    X_scaled_all = scaler.fit_transform(X)
    cv_acc = cross_val_score(clf, X_scaled_all, y, cv=cv, scoring='accuracy').mean()

    cm = confusion_matrix(y_test, y_pred)

    results.append({
        "Features (k)": len(features),
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1 Score": round(f1, 4),
        "ROC-AUC": round(roc_auc, 4),
        "5-Fold CV Accuracy": round(cv_acc, 4),
        "Confusion Matrix [TN, FP, FN, TP]": list(cm.ravel())
    })

    trained_models[name] = (clf, features)


In [ ]:

# 3. Display Performance Table
t6_performance_matrix = pd.DataFrame(results, index=feature_subsets.keys())

print("=== T6: LOGISTIC REGRESSION PERFORMANCE MATRIX ===")
print(t6_performance_matrix.to_markdown())

# Detailed Report for the Full Model
best_clf, best_feats = trained_models["Full (+ Pool, Security, Parking, Lift)"]
X_best = df[best_feats].fillna(df[best_feats].mean())
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_best, y, test_size=0.2, random_state=42, stratify=y)
scaler_b = StandardScaler()
X_test_b_scaled = scaler_b.fit(X_train_b).transform(X_test_b)
y_pred_best = best_clf.predict(X_test_b_scaled)

print("\n=== CLASSIFICATION REPORT (FULL FEATURE MODEL) ===")
print(classification_report(y_test_b, y_pred_best, target_names=["New (0)", "Resale (1)"], zero_division=0))

=== T6: LOGISTIC REGRESSION PERFORMANCE MATRIX ===
|                                        |   Features (k) |   Accuracy |   Precision |   Recall |   F1 Score |   ROC-AUC |   5-Fold CV Accuracy | Confusion Matrix [TN, FP, FN, TP]                              |
|:---------------------------------------|---------------:|-----------:|------------:|---------:|-----------:|----------:|---------------------:|:---------------------------------------------------------------|
| Core (Area, Price)                     |              2 |     0.6187 |      0.5065 |   0.0155 |     0.0301 |    0.5974 |               0.6196 | [np.int64(4040), np.int64(38), np.int64(2476), np.int64(39)]   |
| Extended (+ BHK, Gym)                  |              4 |     0.6202 |      0.5534 |   0.0227 |     0.0435 |    0.5812 |               0.621  | [np.int64(4032), np.int64(46), np.int64(2458), np.int64(57)]   |
| Full (+ Pool, Security, Parking, Lift) |              8 |     0.6243 |      0.545  |   0.0915 |     0.1